# Time-Series Causal Discovery & Gradual Pattern Benchmark

This notebook implements a comprehensive benchmark evaluating our proposed frameworks (**GRAANK** and **T-GRAANK**) against industry-standard causal inference baselines.
 We evaluate performance across two modern standards:
 1. **TimeGraph (Synthetic)**: For testing precise time-lags and non-linear patterns.
 2. **CausalRivers (Real-World Spatiotemporal)**: For scale and geographical directionality.


In [6]:
# Import libraries

import pandas as pd

## Import from local directory
import sys
sys.path.insert(0, '.')

from Synthetic.benchmark import generate_causalrivers_mock, generate_timegraph_mock, evaluate_predictions
from Synthetic.baseline_methods import run_classical_statistics, run_granger_causality, run_pcmci_tigramite, run_pc_algorithm, run_t_graank

print("All libraries successfully imported!")

All libraries successfully imported!


## 1. TimeGraph (Synthetic Dataset) Benchmarking

### Benchmarking Procedure
* We simulate the exact structural data properties expected by the **TimeGraph** and **CausalRivers** APIs, ensuring we have adjacency matrices for the ground-truth graphs.

* To maintain high engineering standards, we wrap every baseline to output a standard 2x2 binary adjacency matrix representing whether a causal/gradual relationship was detected.

* This benchmack engine loops through both datasets, records execution outcomes, and calculates standard validation metrics.

In [7]:

datasets = {
    "TimeGraph (Synthetic)": generate_timegraph_mock(),
    "CausalRivers (Real-World)": generate_causalrivers_mock()
}

results_master = []

for dataset_name, (df, ground_truth) in datasets.items():
    # print(f"\nEvaluating Frameworks on: {dataset_name}...")
    # print(f"{df.head()}\n")

    algorithms = {
        "Classical Statistics": run_classical_statistics(df),
        "Granger Causality": run_granger_causality(df),
        "Tigramite / PCMCI": run_pcmci_tigramite(df),
        "PC Algorithm": run_pc_algorithm(df),
        # "GRAANK (Our Baseline)": run_graank_mock(df),
        "T-GRAANK (Our Proposed)": run_t_graank(df)
    }

    for algo_name, pred_matrix in algorithms.items():
        precision, recall, f1 = evaluate_predictions(ground_truth, pred_matrix)
        results_master.append({
            "Dataset": dataset_name,
            "Algorithm": algo_name,
            "Precision": round(precision, 2),
            "Recall": round(recall, 2),
            "F1-Score": round(f1, 2)
        })

C:\owuor_lab\GP-Mining\.venv_gp\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


  0%|          | 0/2 [00:00<?, ?it/s]

C:\owuor_lab\GP-Mining\.venv_gp\Lib\site-packages\statsmodels\tsa\stattools.py:1556: FutureWarning: verbose is deprecated since functions should not print results
  warnings.warn(


  0%|          | 0/2 [00:00<?, ?it/s]

### (Synthetic) Benchmark Summary

In [8]:
df_results = pd.DataFrame(results_master)
# Pivot for modern academic reporting alignment
df_pivot = df_results.pivot(index="Algorithm", columns="Dataset", values=["Precision", "Recall", "F1-Score"])
df_pivot

Precision                        \
Dataset                 CausalRivers (Real-World) TimeGraph (Synthetic)   
Algorithm                                                                 
Classical Statistics                          0.5                   0.0   
Granger Causality                             1.0                   1.0   
PC Algorithm                                  1.0                   0.0   
T-GRAANK (Our Proposed)                       1.0                   1.0   
Tigramite / PCMCI                             1.0                   1.0   

                                           Recall                        \
Dataset                 CausalRivers (Real-World) TimeGraph (Synthetic)   
Algorithm                                                                 
Classical Statistics                          1.0                   0.0   
Granger Causality                             1.0                   1.0   
PC Algorithm                                  1.0                   0.0   
T-GRAANK (Our Proposed)                       1.0                   1.0   
Tigramite / PCMCI                             1.0                   1.0   

                                         F1-Score                        
Dataset                 CausalRivers (Real-World) TimeGraph (Synthetic)  
Algorithm                                                                
Classical Statistics                         0.67                   0.0  
Granger Causality                            1.00                   1.0  
PC Algorithm                                 1.00                   0.0  
T-GRAANK (Our Proposed)                      1.00                   1.0  
Tigramite / PCMCI                            1.00                   1.0

## CausalRiver (real dataset) Benchmarking

In [15]:
### Load the raw predictions and format them accordingly:
result_paths = [
    "CausalRiver/results/tgraank_close_3/20260821124601/scoring.csv",
    #"CausalRiver/results/tgraank_close_5/20260821124601/scoring.csv",
    "CausalRiver/results/tgraank_root_cause_3/20260825173222/scoring.csv",
    #"CausalRiver/results/tgraank_root_cause_5/20260825173222/scoring.csv",
    "CausalRiver/results/tgraank_1_random_3/20260819161921/scoring.csv",
    #"CausalRiver/results/tgraank_1_random_5/20260819161921/scoring.csv",
    "CausalRiver/results/tgraank_confounder_3/20260818142651/scoring.csv",
    #"CausalRiver/results/tgraank_confounder_5/20260804200814/scoring.csv",
    "CausalRiver/results/tgraank_random_3/20260825054633/scoring.csv",
    #"CausalRiver/results/tgraank_random_5/20260825054633/scoring.csv",
    #"CausalRiver/results/tgraank_disjoint_10/20260811182452/scoring.csv",
]


df_scorings = pd.concat([pd.read_csv(p, index_col=0).loc[["Individual AUROC"]] for p in result_paths])
df_scorings.index = ["Close 3", "Root cause 3", "Random+1 3", "Confounder 3", "Random 3"]
df_scorings.columns = ["TGRAANK"]
df_scorings.T

#df_scorings.T.to_csv("tgraank_submission.csv")

# "Close 3" "Close 5" "Root cause 3" "Root cause 5" "Random+1 3" "Random+1 5" "Confounder 3" "Confounder 5" "Random 3" "Random 5" "Disjoint 10"

,Close 3,Root cause 3,Random+1 3,Confounder 3,Random 3
TGRAANK,0.677891,0.7008,0.828725,0.651331,0.669843
